# 04 - Feature Selection (3-Layer Pipeline)

Layer 1: Variance + Mutual Information on raw genes

Layer 2: Domain-specific feature engineering with correlation filtering

Layer 3: Binary PSO assembly over genes plus engineered features

All logic lives in `src/feature_selection.py`. This notebook only loads data, detects clinical columns, calls the pipeline, and saves artifacts.

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import joblib
import config
from src.io import logger
from src.feature_selection import run_3layer_feature_selection

## Step 1: Load Preprocessed Training Data

In [2]:
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv", index_col=0)

y_train_df = pd.read_csv(config.PROCESSED_DIR / "y_train.csv")
y_train = y_train_df.iloc[:, 0] if len(y_train_df.columns) == 1 else y_train_df["BCR"]

logger.info(f"Training data loaded: {X_train.shape}, Positives: {int(y_train.sum())}")

2026-09-01 01:08:49 | INFO     | prostate_bcr | Training data loaded: (343, 19018), Positives: 46


## Step 2: Detect Clinical / Engineered Columns

These columns are excluded from Layer 1 gene filtering and passed explicitly to `run_3layer_feature_selection` as `clinical_cols`.

In [3]:
clinical_keywords = [
    "gleason", "margin", "lymph", "tumor stage", "psa",
    "bone scan", "cause of death", "ct scan", "primary therapy",
    "age", "race", "ethnicity", "weight", "height",
    "mri", "icd-o", "histology", "patient primary", "diagnosis",
    "year cancer", "radical prostatectomy", "adjuvant", "radiation",
    "hormone", "chemotherapy", "surgery", "metastasis", "recurrence",
    "pathway_score", "_score", "risk", "total", "ratio", "balance",
]

clinical_cols = [
    c for c in X_train.columns
    if any(kw.lower() in c.lower() for kw in clinical_keywords)
]

logger.info(f"Detected {len(clinical_cols)} clinical/engineered columns to exclude from Layer 1.")
print(f"Clinical/engineered columns: {len(clinical_cols)}")
print(f"Gene columns:                {X_train.shape[1] - len(clinical_cols)}")

2026-09-01 01:08:49 | INFO     | prostate_bcr | Detected 132 clinical/engineered columns to exclude from Layer 1.


Clinical/engineered columns: 132
Gene columns:                18886


## Step 3: Run the 3-Layer Feature Selection Pipeline

Fitted exclusively on `X_train` / `y_train`.

In [4]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.TABLES_DIR.mkdir(parents=True, exist_ok=True)

fitted_l1 = None
final_features = []

try:
    fitted_l1, final_features = run_3layer_feature_selection(
        X_train=X_train,
        y_train=y_train,
        clinical_cols=clinical_cols,
        run_pso=True,
        random_state=config.RANDOM_STATE,
    )

    print("Pipeline complete.")
    print(f"  Layer 1 (genes):       {len(fitted_l1['mi_features'])} MI-selected genes")
    print(f"  Final total features:  {len(final_features)}")

except ValueError as e:
    print("CRITICAL ERROR: pipeline failed during fit or transform.")
    print(f"  Error details: {e}")

    if "feature names should match" in str(e).lower():
        # Actionable debug info: show which genes were present at fit
        # time but are missing now (usually rare-category one-hot columns
        # that were dropped for this particular train/test split).
        if fitted_l1 is not None and fitted_l1.get("is_fitted"):
            fit_cols = set(
                fitted_l1["vt"].get_feature_names_out().tolist()
                if hasattr(fitted_l1["vt"], "get_feature_names_out")
                else []
            )
            current_cols = set(X_train.columns)
            missing_now = sorted(fit_cols - current_cols)
            print(f"  {len(missing_now)} columns present at fit time are now missing:")
            for col in missing_now[:15]:
                print(f"    - {col}")
            print("  FIX: ensure the train/test one-hot encoding is built from a "
                "shared category list (fit once, applied to both splits), or "
                "reindex X_train to include all fit-time columns before transform.")
    else:
        raise

except Exception as e:
    print(f"UNEXPECTED ERROR: {type(e).__name__}: {e}")
    raise

2026-09-01 01:09:21 | INFO     | prostate_bcr | Layer 1 - Selected 200 genes from 18886 raw genes


CRITICAL ERROR: pipeline failed during fit or transform.
  Error details: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Tumor Other Histologic Subtype_25-30% ductal component
- Tumor Other Histologic Subtype_Adenocarcinoma prostate with prominent ductal differentiation identified
- Tumor Other Histologic Subtype_Mixed
- Tumor Other Histologic Subtype_Mixed ductal (65%) and Acinar
- Tumor Other Histologic Subtype_Prostate Adenocarcinoma, Not Otherwised Specified, with ductal featues
- ...



## Step 4: Save Artifacts

In [6]:
if final_features:
    joblib.dump(fitted_l1, config.MODELS_DIR / "fitted_layer1_selector.joblib")
    print(f"Saved Layer 1 selector to: {config.MODELS_DIR / 'fitted_layer1_selector.joblib'}")

    pd.DataFrame({"feature": final_features}).to_csv(
        config.TABLES_DIR / "selected_features_final.csv", index=False
    )
    print(
        f"Saved feature list ({len(final_features)} items) to: "
        f"{config.TABLES_DIR / 'selected_features_final.csv'}"
    )
else:
    print("WARNING: no features were generated. Nothing was saved.")
    print("Resolve the error above and re-run this notebook.")

Resolve the error above and re-run this notebook.
